In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv()

llm = ChatGoogleGenerativeAI(
    model=os.getenv("GEMINI_MODEL", "gemini-2.0-flash"),
    google_api_key=os.getenv("GEMINI_API_FREE_KEY") or os.getenv("GEMINI_API_KEY")
)

llm.invoke([HumanMessage("잘 지냈어?")])

AIMessage(content=[{'type': 'text', 'text': '네, 잘 지냈어요! 물어봐 주셔서 고마워요. 😊\n\n오늘 하루는 어떠셨나요? 기분 좋은 일이 있었는지, 아니면 제가 도와드릴 만한 일이 있는지 궁금하네요. 무엇이든 편하게 이야기해 주세요!', 'extras': {'signature': 'EvQLCvELAb4+9vvSi8D1sTRxqXOhwlyy7lyV4WXfxAxW8LJqVJa0N9/oLGNG/h08aE/1QEWdmgLrhxxQGrRP4uvheXBSKST4JD40GNUC9V3hoY68xkxe9SGgULyy5RIqumKA49cxrAO8/5iDr+IudrNjCXBVpLDXlpSSk9qDUtbvXahIYN9VFTcv7AEkhYxSJWG1hFR3DXMxkoQjikhY/QsdP8cQAripdN5Drq6vmNFEnhDi+81d77JjiOMoE+Ah77PMHz4LCttGcMzxCV2YRNpfRREUGllBoU2qcBxsIfL1TVsSEmcca5Lu0WRafwiHjMdm/ivp/jw8XxbmSj7sJ3YFboyj14NtwahY5eaQFi9EPZClRIBPqfLiMexHydspFT2zW0Sju2U1PGza6CmMzqVhitRMdoW1fV99Kq0kaQQ48G8PeL2/lfWHfcwk/qzRTtatVBddr5pf1CJBMV9JLoli8sCyfHIHbY7jiThPEEo/dbJ3j1OFhv9RJyjs8VqkmxTZWhg2EgHT/sZC1eccyd0nR00CM1tGhTI3HF7Ac/NtFjDaThjH5x+QNKX6+CKh6d2ajVHFnBOMv3Y9y/td/VD/AkOOnOfiB6Fm3ErSUrtmfbrlLC8lfY0OLYFLRT68XsoVEe8r3eusM8HPbuGTByJCeNtqBLoUmMrhi51kLzmT3ozTRok8UIqWrEsliPJOzHPI1EuyBO2gyRncJAUdDMDkxG1/GdnvuR6SHP8BARWEjYDBszhDN29R+aX75FacptLIdm5cwnyZEEUM2qtIk2NqnJohC2/GIDgEQB+2N6eGkMXwFv2Ox

In [5]:
from langchain_core.tools import tool
from datetime import datetime
import pytz

@tool # @tool 데코레이터를 사용하여 함수를 도구로 등록
def get_current_time(timezone: str, location: str) -> str:
    """ 현재 시각을 반환하는 함수

    Args:
        timezone (str): 타임존 (예: 'Asia/Seoul') 실제 존재하는 타임존이어야 함
        location (str): 지역명. 타임존이 모든 지명에 대응되지 않기 때문에 이후 llm 답변 생성에 사용됨
    """
    tz = pytz.timezone(timezone)
    now = datetime.now(tz).strftime("%Y-%m-%d %H:%M:%S")
    location_and_local_time = f'{timezone} ({location}) 현재시각 {now} ' # 타임존, 지역명, 현재시각을 문자열로 반환
    print(location_and_local_time)
    return location_and_local_time


In [6]:
# 도구를 tools 리스트에 추가하고, tool_dict에도 추가
tools = [get_current_time,]
tool_dict = {"get_current_time": get_current_time,}

# 도구를 모델에 바인딩: 모델에 도구를 바인딩하면, 도구를 사용하여 llm 답변을 생성할 수 있음
llm_with_tools = llm.bind_tools(tools)

In [7]:
from langchain_core.messages import SystemMessage

# (4) 사용자의 질문과 tools 사용하여 llm 답변 생성
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

# (5) llm_with_tools를 사용하여 사용자의 질문에 대한 llm 답변 생성
response = llm_with_tools.invoke(messages)
messages.append(response)

# (6) 생성된 llm 답변 출력
print(messages)

[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}), HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}), AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"location": "\\ubd80\\uc0b0", "timezone": "Asia/Seoul"}'}, '__gemini_function_call_thought_signatures__': {'f7c3993e-c005-4d70-933f-df595255b77b': 'EsMDCsADAb4+9vtPcVBnD8gINzPeDGHm37hmzddfT9/Qhme/LLw00eGPUFk1D1PNjXOCOA5HGjhHaIzMDJr28LIMpIbVNywiJBB64hXuXZ702sX/SBKiK1HcZgvoJD1IosNk1+oE/NdjtrN6M9TXnXjTmnBVBNHXB73C8jDknOYcjdDGnbPecqNl2UyZnbg5z7vSEM8ro2vUdLDb5M2FE3Jj1Q1idnSJkNZY0tHpQcrsqi7uX05/5oMUzf2YMdSyM8zotgT67cDpGEbwZKCrGjO26RtFi41sUaP6T50BKo4jSWVZCwh54hfl760Xg/Xir1YciI9V5JNHIfwwYi47B06bkgV0xgYai1H5tCBoFRvzl1+0uBAFmBV04h8v7spTU8gLbwzXgIm0uKQ4jtU1QyDllcrnntgd1Ns4oNy9HQp1Ph62t9Gb3Hno5vMu9rrMBii7WNh4G+EEf8nAqJ8I7zBfjfM9g4j6BT7AQBVWzyDRhNT+fNN4AFDp3KwPzvEn6Z3ixK35E7VV9n1tOCdL62061W3FoR/Z4UIRjV9Ot49QK49

In [8]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # (7) tool_dict를 사용하여 도구 함수를 선택
    print(tool_call["args"]) # (8) 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # (9) 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': '부산', 'timezone': 'Asia/Seoul'}
Asia/Seoul (부산) 현재시각 2026-03-23 10:53:39 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessage(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"location": "\\ubd80\\uc0b0", "timezone": "Asia/Seoul"}'}, '__gemini_function_call_thought_signatures__': {'f7c3993e-c005-4d70-933f-df595255b77b': 'EsMDCsADAb4+9vtPcVBnD8gINzPeDGHm37hmzddfT9/Qhme/LLw00eGPUFk1D1PNjXOCOA5HGjhHaIzMDJr28LIMpIbVNywiJBB64hXuXZ702sX/SBKiK1HcZgvoJD1IosNk1+oE/NdjtrN6M9TXnXjTmnBVBNHXB73C8jDknOYcjdDGnbPecqNl2UyZnbg5z7vSEM8ro2vUdLDb5M2FE3Jj1Q1idnSJkNZY0tHpQcrsqi7uX05/5oMUzf2YMdSyM8zotgT67cDpGEbwZKCrGjO26RtFi41sUaP6T50BKo4jSWVZCwh54hfl760Xg/Xir1YciI9V5JNHIfwwYi47B06bkgV0xgYai1H5tCBoFRvzl1+0uBAFmBV04h8v7spTU8gLbwzXgIm0uKQ4jtU1QyDllcrnntgd1Ns4oNy9HQp1Ph62t9Gb3Hno5vMu9rrMBii7WNh4G+EEf8nAqJ8I7zBfjfM9g4j6BT7AQBVWzyDRhNT+fNN4AFDp3KwPzvEn6Z3ixK35E7VV9n1tOCdL62061W3FoR/Z4UIRjV9Ot49QK

In [9]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': '부산은 현재 2026년 3월 23일 오전 10시 53분입니다.'}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d1865-dff0-7300-a34e-7911ef1429e3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 325, 'output_tokens': 27, 'total_tokens': 352, 'input_token_details': {'cache_read': 0}})

In [10]:
from pydantic import BaseModel, Field

class StockHistoryInput(BaseModel):
    ticker: str = Field(..., title="주식 코드", description="주식 코드 (예: AAPL)")
    period: str = Field(..., title="기간", description="주식 데이터 조회 기간 (예: 1d, 1mo, 1y)")


In [11]:
import yfinance as yf

@tool
def get_yf_stock_history(stock_history_input: StockHistoryInput) -> str:
    """ 주식 종목의 가격 데이터를 조회하는 함수"""
    stock = yf.Ticker(stock_history_input.ticker)
    history = stock.history(period=stock_history_input.period)
    history_md = history.to_markdown() 

    return history_md

tools = [get_current_time, get_yf_stock_history]
tool_dict = {"get_current_time": get_current_time, "get_yf_stock_history": get_yf_stock_history}

llm_with_tools = llm.bind_tools(tools)

In [12]:
messages.append(HumanMessage("테슬라는 한달 전에 비해 주가가 올랐나 내렸나?"))

response = llm_with_tools.invoke(messages)
print(response)
messages.append(response)

content=[] additional_kwargs={'function_call': {'name': 'get_yf_stock_history', 'arguments': '{"stock_history_input": {"ticker": "TSLA", "period": "1mo"}}'}, '__gemini_function_call_thought_signatures__': {'a845ff54-cae9-43fb-841a-f9e4a4ed99bb': 'EsYCCsMCAb4+9vsOrJBYW7CXXuj5eFXa+L07Vjv/Bazvl1f2cYXWj8PHYskXDBm5vgGwsOpYSKr+UmsyUJntmOlcZCBI5XyXLjT6DRw+QSRBMptRkZiN2YB7HFnzYwgu0tXzQuQp/PVE7PR8YeUbyPQtkp2C/65LmZzB55817kYZ7PdfiPXEQaf0V5yCrRuWawWByFOzlYKkviwkkVbxy1Oho/LXxvrPNbqknO4MLo9stwyzDrzIwtg1bgpyUUVm+yEbpTR1kWRSvyjuTMKfJ4pmbPtvH9w9n6CoAiwBSs4JlQoW4fGdTQusJ0sJDMGOyEUbzpLxVYpNhWnjbWyVUVpgnsoq0jnmPloP4E35+r0U4ut77Frhzc1+cfG/puJ9vLkNQkuaiOBB6byOLaJ+IPT0Xj5NG7J3KcXuqY3y37/1/u4M2maQrJo='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--019d1866-0240-74a3-93da-f6f6c23bf7ae-0' tool_calls=[{'name': 'get_yf_stock_history', 'args': {'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}, 'i

In [13]:
for tool_call in response.tool_calls:
    selected_tool = tool_dict[tool_call["name"]]
    print(tool_call["args"])
    tool_msg = selected_tool.invoke(tool_call)
    messages.append(tool_msg)
    print(tool_msg)

{'stock_history_input': {'ticker': 'TSLA', 'period': '1mo'}}
content='| Date                      |   Open |   High |    Low |   Close |      Volume |   Dividends |   Stock Splits |\n|:--------------------------|-------:|-------:|-------:|--------:|------------:|------------:|---------------:|\n| 2026-02-23 00:00:00-05:00 | 407.29 | 407.7  | 394.04 |  399.83 | 6.968e+07   |           0 |              0 |\n| 2026-02-24 00:00:00-05:00 | 399.5  | 410.82 | 397.64 |  409.38 | 5.85795e+07 |           0 |              0 |\n| 2026-02-25 00:00:00-05:00 | 412.15 | 420.34 | 412.15 |  417.4  | 5.48097e+07 |           0 |              0 |\n| 2026-02-26 00:00:00-05:00 | 414.42 | 416.81 | 403.66 |  408.58 | 5.36025e+07 |           0 |              0 |\n| 2026-02-27 00:00:00-05:00 | 402.94 | 407.12 | 398.11 |  402.51 | 5.68901e+07 |           0 |              0 |\n| 2026-03-02 00:00:00-05:00 | 390.6  | 404.54 | 388.25 |  403.32 | 5.50883e+07 |           0 |              0 |\n| 2026-03-03 00:00:00-05:0

In [14]:
llm_with_tools.invoke(messages)

AIMessage(content=[{'type': 'text', 'text': '테슬라(TSLA) 주가는 한 달 전과 비교했을 때 **하락**했습니다.\n\n*   **한 달 전(2026년 2월 23일) 종가:** $399.83\n*   **최근(2026년 3월 20일) 종가:** $367.96\n\n약 한 달 사이에 주가가 약 **$31.87 (약 8%) 하락**한 것으로 나타납니다.', 'extras': {'signature': 'EvcCCvQCAb4+9vtmP3rzWhQmrgW8BcEKXcrQXJpQ/oL0xeRWlVVKvMWqYb2Ynrc/vAJdZ6SmoCbHncZ1FsZIZ2VR0F3BvnpjJqws/IVKNQ8GIBfwQtqtpfaVd5E/4Y6HVD2LlWsd3O83JQJUW6eCNBjPlYdUeQiyjYHPU+zPOoB+/99yFXNE06GONy5u81xkXYMyE7aA8dg5xOkbbbOULMCFeL680ByzDFN6RzYHJtUNBUOkiVn3oXyG/uBqXwMrP1/OOt91jJknegiCsCuhr1yKTDryPzeGwH+QSg3cV3IPByXk0KQOfEtC/tD8I0ZXXurk1/dmrMIdoBP4rglAQvH1iQzJ1VTnvQBRFkg/4TnDVQZPzKeC+zn0M8Xitk1ronkuqSq0GVol0oovmrPv5RB/XzZT9Blo57bPl12+hgVfh3+g+gGgh8pyyBhD0vd0nr7yt3HUiSZY7qEFBdfEYer0Ep2VpnZxJee0pCq03wIuUm9JWbGNR3XF'}}], additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019d1866-117f-7a92-8aef-07512c9a38f4-0', tool_calls=[], invalid_tool_c

## 08-4 스트림 출력

In [3]:
for c in llm.stream([HumanMessage("잘 지냈어? 한국 사회의 문제점에 대해 이야기해줘.")]):
    print(c.content, end='|') 

[{'type': 'text', 'text': '네, 덕분에 잘 지내고 있었습니다! 물어봐 주셔서', 'index': 0}]|[{'type': 'text', 'text': ' 감사합니다.\n\n한국 사회는 짧은 기간 동안 눈부신 경제 성장을 이루었지만, 그 이면에는 여러', 'index': 0}]|[{'type': 'text', 'text': ' 가지 복합적인 사회적 문제들이 자리 잡고 있습니다. 현재 한국 사회가 직면한 주요 문제점들을 몇', 'index': 0}]|[{'type': 'text', 'text': ' 가지 핵심 키워드로 정리해 드릴게요.\n\n### 1. 저출산 및 고령화 문제 (인구 절', 'index': 0}]|[{'type': 'text', 'text': '벽)\n가장 시급하고 치명적인 문제입니다. 한국의 합계출산율은 세계 최저', 'index': 0}]|[{'type': 'text', 'text': ' 수준(0.7명대 이하)으로 떨어졌습니다.\n*   **원인:** 주거비 부담', 'index': 0}]|[{'type': 'text', 'text': ', 양육비 및 사교육비 부담, 경력 단절에 대한 두려움, 가치관의', 'index': 0}]|[{'type': 'text', 'text': ' 변화 등.\n*   **결과:** 노동 인구 감소, 연금 고갈 우려, 국가 경쟁', 'index': 0}]|[{'type': 'text', 'text': '력 하락 등으로 이어지고 있습니다.\n\n### 2. 부동산 문제와 경제적 불평등\n자산의', 'index': 0}]|[{'type': 'text', 'text': ' 대부분이 부동산에 쏠려 있고, 집값 상승으로 인해 근로 소득만으로는 내 집 마련이', 'index': 0}]|[{'type': 'text', 'text': ' 거의 불가능해진 상황입니다.\n*   **양극화:** 자산가와 무주택자', 'index': 0}]|[{'type': 'text', 'text': ' 사이의 격차가 벌어지며 상

In [15]:
messages = [
    SystemMessage("너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다."),
    HumanMessage("부산은 지금 몇시야?"),
]

response = llm_with_tools.stream(messages)

# 파편화된 tool_call 청크를 하나로 합치기 
is_first = True
for chunk in response:    
    print("chunk type: ", type(chunk))
    
    if is_first:
        is_first = False
        gathered = chunk
    else:
        gathered += chunk
    
    print("content: ", gathered.content, "tool_call_chunk", gathered.tool_calls)

messages.append(gathered)

chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [] tool_call_chunk [{'name': 'get_current_time', 'args': {'location': '부산', 'timezone': 'Asia/Seoul'}, 'id': '4c68add4-8dfc-49ab-a621-87842156f181', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [] tool_call_chunk [{'name': 'get_current_time', 'args': {'location': '부산', 'timezone': 'Asia/Seoul'}, 'id': '4c68add4-8dfc-49ab-a621-87842156f181', 'type': 'tool_call'}]
chunk type:  <class 'langchain_core.messages.ai.AIMessageChunk'>
content:  [] tool_call_chunk [{'name': 'get_current_time', 'args': {'location': '부산', 'timezone': 'Asia/Seoul'}, 'id': '4c68add4-8dfc-49ab-a621-87842156f181', 'type': 'tool_call'}]


In [16]:
gathered

AIMessageChunk(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"location": "\\ubd80\\uc0b0", "timezone": "Asia/Seoul"}'}, '__gemini_function_call_thought_signatures__': {'4c68add4-8dfc-49ab-a621-87842156f181': 'EsECCr4CAb4+9vsZinL4G3YDa+k1e82LsUzbpYehVWKqILsL2O365NYCqtUJ+ZdlrJZDKHA1iVUu49LndaMKsFxtJ/IEk+RsytGbawvN0n6pblx4wcSJBWfTDC7qzmChsGY4cfMe+1W4F4zrU9Z6QaakbNWOnCd0yxX27LcvIZjf06jF7GtnWbsOaImhK/9TQtsfq7mQJJYQfQwEACi++jrEpaEqHlBrMLN6kPACPkkHuTTxIf7V74E0jpkA2UP53yrsQm2Mwi16oJbvmWHWfDARisLtOAdpTUu+D1Csd8KHyQASKG5RXRWGxlYTWG5XMRuPtPqxz+u0vNmYH8miQx80eiohBLSHwoVlRt0Utb9w9pEAjhfX7CUmFr7qmePhY9D5ZoCuJnhVRqbCOvggRgbgxxH8xn5WPM08OC8huq343vBc'}}, response_metadata={'safety_ratings': [], 'model_provider': 'google_genai', 'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview'}, id='lc_run--019d1866-1bf4-75d0-8202-1117ac612f6e', tool_calls=[{'name': 'get_current_time', 'args': {'location': '부산', 'timezone': 'Asia/Seoul'}, 'id': '4c68add4-8dfc-

In [17]:
for tool_call in gathered.tool_calls:
    selected_tool = tool_dict[tool_call["name"]] # tool_dict를 사용하여 도구 이름으로 도구 함수를 선택
    print(tool_call["args"]) # 도구 호출 시 전달된 인자 출력
    tool_msg = selected_tool.invoke(tool_call) # 도구 함수를 호출하여 결과를 반환
    messages.append(tool_msg)

messages

{'location': '부산', 'timezone': 'Asia/Seoul'}
Asia/Seoul (부산) 현재시각 2026-03-23 10:54:27 


[SystemMessage(content='너는 사용자의 질문에 답변을 하기 위해 tools를 사용할 수 있다.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='부산은 지금 몇시야?', additional_kwargs={}, response_metadata={}),
 AIMessageChunk(content=[], additional_kwargs={'function_call': {'name': 'get_current_time', 'arguments': '{"location": "\\ubd80\\uc0b0", "timezone": "Asia/Seoul"}'}, '__gemini_function_call_thought_signatures__': {'4c68add4-8dfc-49ab-a621-87842156f181': 'EsECCr4CAb4+9vsZinL4G3YDa+k1e82LsUzbpYehVWKqILsL2O365NYCqtUJ+ZdlrJZDKHA1iVUu49LndaMKsFxtJ/IEk+RsytGbawvN0n6pblx4wcSJBWfTDC7qzmChsGY4cfMe+1W4F4zrU9Z6QaakbNWOnCd0yxX27LcvIZjf06jF7GtnWbsOaImhK/9TQtsfq7mQJJYQfQwEACi++jrEpaEqHlBrMLN6kPACPkkHuTTxIf7V74E0jpkA2UP53yrsQm2Mwi16oJbvmWHWfDARisLtOAdpTUu+D1Csd8KHyQASKG5RXRWGxlYTWG5XMRuPtPqxz+u0vNmYH8miQx80eiohBLSHwoVlRt0Utb9w9pEAjhfX7CUmFr7qmePhY9D5ZoCuJnhVRqbCOvggRgbgxxH8xn5WPM08OC8huq343vBc'}}, response_metadata={'safety_ratings': [], 'model_provider': 'google_genai', 'finish_reason': 'STOP', 'model_name': '

In [18]:
for c in llm_with_tools.stream(messages):
    print(c.content, end='|')

[{'type': 'text', 'text': '부산의 현재 시간', 'index': 0}]|[{'type': 'text', 'text': '은 2026년 3월 23일 오전 10시', 'index': 0}]|[{'type': 'text', 'text': ' 54분입니다.', 'index': 0}]|[]|[]|